# Downloading and Reading the CSI Dataset

In [20]:
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import json

ANTENNACOUNT = 32

def load_and_calibrate(path, offset_path):
	offsets = None
	with open(offset_path, "r") as offsetfile:
		offsets = json.load(offsetfile)
	
	def record_parse_function(proto):
		record = tf.io.parse_single_example(
			proto,
			{
				"csi": tf.io.FixedLenFeature([], tf.string, default_value=""),
				"pos-tachy": tf.io.FixedLenFeature([], tf.string, default_value=""),
				"time": tf.io.FixedLenFeature([], tf.float32, default_value=0),
			},
		)

		csi = tf.ensure_shape(tf.io.parse_tensor(record["csi"], out_type=tf.float32), (ANTENNACOUNT, 1024, 2))
		csi = tf.complex(csi[:, :, 0], csi[:, :, 1])
		csi = tf.signal.fftshift(csi, axes=1)

		position = tf.ensure_shape(tf.io.parse_tensor(record["pos-tachy"], out_type=tf.float64), (3))
		time = tf.ensure_shape(record["time"], ())

		return csi, position[:2], time

	def apply_calibration(csi, pos, time):
		sto_offset = tf.tensordot(tf.constant(offsets["sto"]), 2 * np.pi * tf.range(tf.shape(csi)[1], dtype = np.float32) / tf.cast(tf.shape(csi)[1], np.float32), axes = 0)
		cpo_offset = tf.tensordot(tf.constant(offsets["cpo"]), tf.ones(tf.shape(csi)[1], dtype = np.float32), axes = 0)
		csi = tf.multiply(csi, tf.exp(tf.complex(0.0, sto_offset + cpo_offset)))

		return csi, pos, time

	return tf.data.TFRecordDataset(path).map(record_parse_function, num_parallel_calls = tf.data.AUTOTUNE).map(apply_calibration, num_parallel_calls = tf.data.AUTOTUNE)

inputpaths = [
    {
        "tfrecords": "datasets/dichasus-cf02.tfrecords",
        "offsets": "datasets/reftx-offsets-dichasus-cf02.json",
    },
    {
        "tfrecords": "datasets/dichasus-cf03.tfrecords",
        "offsets": "datasets/reftx-offsets-dichasus-cf03.json",
    },
    {
        "tfrecords": "datasets/dichasus-cf04.tfrecords",
        "offsets": "datasets/reftx-offsets-dichasus-cf04.json",
    },
]

training_set = load_and_calibrate(inputpaths[0]["tfrecords"], inputpaths[0]["offsets"])

for path in inputpaths[1:]:
	training_set = training_set.concatenate(load_and_calibrate(path["tfrecords"], path["offsets"]))

# Decimate dataset: Use only every 4th datapoint (to reduce number of points)
training_set = training_set.enumerate().filter(lambda idx, value : (idx % 4 == 0))
training_set = training_set.map(lambda idx, value : value)
num_samples = sum(1 for _ in training_set)
print(num_samples)

20997


In [21]:
# Reproducibility for model initialization and triplet sampling.
SEED = 10
import random
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Feature Engineering

In [22]:
def average_over_subcarriers(csi, pos, time):
	csi = tf.reduce_mean(csi[:, 508:516], axis=1)
	return csi, pos, time

training_set = training_set.map(average_over_subcarriers, num_parallel_calls = tf.data.AUTOTUNE)

def scale_r2m(csi, pos, time):
	beta = 1 + 1 / 16
	
	second_moment = tf.tensordot(csi, tf.math.conj(csi), axes=0)
	csi = tf.constant(ANTENNACOUNT**(beta - 1), dtype = tf.complex64) / tf.math.pow(tf.norm(second_moment, ord = "fro", axis = (0, 1)), beta) * second_moment
	csi = tf.squeeze(tf.reshape(csi, (1, ANTENNACOUNT * ANTENNACOUNT)), axis = 0)
	csi = tf.math.real(csi)

	return csi, pos, time

training_set = training_set.map(scale_r2m, num_parallel_calls = tf.data.AUTOTUNE)

In [23]:
# If you have enough RAM...
training_set = training_set.cache()

# Visualizing the Dataset

In [24]:
# This may take a long time, since TensorFlow has to load the dataset from disk!
groundtruth_positions = []
timestamp_index_map = dict()

for index, data in enumerate(training_set):
	groundtruth_positions.append(data[1].numpy())
	timestamp_index_map[data[2].numpy()] = index

timestamp_index_map = dict(sorted(timestamp_index_map.items()))
groundtruth_positions = np.asarray(groundtruth_positions)

In [25]:
def plot_colorized(positions, title = None):
	# Generate RGB colors for datapoints
	center_point = np.zeros(2, dtype = np.float32)
	center_point[0] = 0.5 * (np.min(groundtruth_positions[:, 0], axis = 0) + np.max(groundtruth_positions[:, 0], axis = 0))
	center_point[1] = 0.5 * (np.min(groundtruth_positions[:, 1], axis = 0) + np.max(groundtruth_positions[:, 1], axis = 0))
	NormalizeData = lambda in_data : (in_data - np.min(in_data)) / (np.max(in_data) - np.min(in_data))
	rgb_values = np.zeros((groundtruth_positions.shape[0], 3))
	rgb_values[:, 0] = 1 - 0.9 * NormalizeData(groundtruth_positions[:, 0])
	rgb_values[:, 1] = 0.8 * NormalizeData(np.square(np.linalg.norm(groundtruth_positions - center_point, axis=1)))
	rgb_values[:, 2] = 0.9 * NormalizeData(groundtruth_positions[:, 1])

	# Plot datapoints
	plt.figure(figsize=(6, 6))
	if title is not None:
		plt.suptitle(title, fontsize=16)
	plt.scatter(positions[:, 0], positions[:, 1], c = rgb_values, s = 5)

plot_colorized(groundtruth_positions, title="Ground Truth Positions")

# Triplet Loss

In [26]:
def triplet_loss(y_true, y_pred):
	anchor, positive, negative = (y_pred[:, :CC_DIMENSIONALITY], y_pred[:, CC_DIMENSIONALITY : 2 * CC_DIMENSIONALITY], y_pred[:, 2 * CC_DIMENSIONALITY :])
	positive_dist = tf.reduce_mean(tf.square(anchor - positive), axis = 1)
	negative_dist = tf.reduce_mean(tf.square(anchor - negative), axis = 1)
			
	return tf.maximum(positive_dist - negative_dist + 1, 0.0)

# Triplet Selection

In [27]:
def generate_positive_sample_lookup(Tc = 1.5):
	lookup = dict()
	suitable_set = dict()
	candidate_iterator = iter(timestamp_index_map.items())
	next_suitable = next(candidate_iterator)

	for anchor_timestamp, anchor_index in timestamp_index_map.items():
		# Add additional later suitable datapoints to suitable_set
		while next_suitable[0] - anchor_timestamp < Tc:
			suitable_set.update((next_suitable,))
			try:
				next_suitable = next(candidate_iterator)
			except StopIteration:
				break

		# Remove too early suitable datapoints from suitable_set
		outdated = []
		for timestamp in suitable_set.keys():
			if anchor_timestamp - timestamp > Tc:
				outdated.append(timestamp)
			else:
				for o in outdated:
					del suitable_set[o]
				break

		# For every potential anchor point, store lookup table of potential positive sample indices
		lookup[anchor_index] = set(suitable_set.values())
		
		# Set of suitable positive samples must not contain anchor itself
		lookup[anchor_index].remove(anchor_index)
		lookup[anchor_index] = list(lookup[anchor_index])

	return lookup

In [28]:
def generate_triplets(nr_of_triplets = 1000, Tc = 1.5):
    print("Generating lookup table for positive samples")
    positive_sample_lookup = generate_positive_sample_lookup(Tc)

    # Generate list of triplet containing *indices* of datapoints in dataset
    triplet_indices = []
    anchor_indices = list(positive_sample_lookup.keys())
    
    while len(triplet_indices) < nr_of_triplets:
        anchor = anchor_indices[np.random.randint(len(anchor_indices))]
        
        # Must ensure that anchor point is not a loner and actually has some close positive samples
        if len(positive_sample_lookup[anchor]) < 1:
            continue

        positive = np.random.choice(positive_sample_lookup[anchor])
        negative = anchor_indices[np.random.randint(len(anchor_indices))]
        
        triplet_indices.append((anchor, positive, negative))

    # Iterate over dataset (on hard drive storage) and load relevant CSI data to "triplets" list (in RAM)
    datapoints_to_load = dict()
    for target, indices in enumerate(triplet_indices):
        for sample in range(3):
            if indices[sample] not in datapoints_to_load:
                datapoints_to_load[indices[sample]] = []
            datapoints_to_load[indices[sample]].append((target, sample))

    datapoints_to_load = dict(sorted(datapoints_to_load.items()))

    anchors = [None for i in range(len(triplet_indices))]
    positives = [None for i in range(len(triplet_indices))]
    negatives = [None for i in range(len(triplet_indices))]

    print("Loading batch of triplets from dataset (on disk or in RAM)")
    for index, data in enumerate(training_set):
        if index in datapoints_to_load:
            for target in datapoints_to_load[index]:
                if target[1] == 0:
                    anchors[target[0]] = data[0]
                elif target[1] == 1:
                    positives[target[0]] = data[0]
                elif target[1] == 2:
                    negatives[target[0]] = data[0]
    print("Finished loading triplet batch")

    return [tf.stack(anchors), tf.stack(positives), tf.stack(negatives)]

# Neural Network Architeinput_anchor = tf.keras.layers.Input(shape=(ANTENNACOUNT * ANTENNACOUNT,))
input_positive = tf.keras.layers.Input(shape=(ANTENNACOUNT * ANTENNACOUNT,))
input_negative = tf.keras.layers.Input(shape=(ANTENNACOUNT * ANTENNACOUNT,))

embedding_anchor = embedding_model(input_anchor)
embedding_positive = embedding_model(input_positive)
embedding_negative = embedding_model(input_negative)

output = tf.keras.layers.concatenate([embedding_anchor, embedding_positive, embedding_negative], axis=1)
model = tf.keras.models.Model([input_anchor, input_positive, input_negative], output)cture

In [29]:
CC_DIMENSIONALITY = 2
embedding_model = tf.keras.models.Sequential(
	[
		tf.keras.layers.Dense(ANTENNACOUNT * ANTENNACOUNT, activation = "relu", input_shape = (ANTENNACOUNT * ANTENNACOUNT,)),
		tf.keras.layers.BatchNormalization(),
		tf.keras.layers.Dense(512, activation = "relu"),
		tf.keras.layers.BatchNormalization(),
		tf.keras.layers.Dense(256, activation = "relu"),
		tf.keras.layers.BatchNormalization(),
		tf.keras.layers.Dense(128, activation = "relu"),
		tf.keras.layers.BatchNormalization(),
		tf.keras.layers.Dense(64, activation = "relu"),
		tf.keras.layers.BatchNormalization(),
		tf.keras.layers.Dense(CC_DIMENSIONALITY, activation = "linear"),
	]
)

d:\anaconda3\envs\cc_MIMO\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [30]:
input_anchor = tf.keras.layers.Input(shape=(ANTENNACOUNT * ANTENNACOUNT,))
input_positive = tf.keras.layers.Input(shape=(ANTENNACOUNT * ANTENNACOUNT,))
input_negative = tf.keras.layers.Input(shape=(ANTENNACOUNT * ANTENNACOUNT,))

embedding_anchor = embedding_model(input_anchor)
embedding_positive = embedding_model(input_positive)
embedding_negative = embedding_model(input_negative)

output = tf.keras.layers.concatenate([embedding_anchor, embedding_positive, embedding_negative], axis=1)
model = tf.keras.models.Model([input_anchor, input_positive, input_negative], output)

# Neural Network Training

In [31]:
optimizer = tf.keras.optimizers.Adam()
model.compile(loss = triplet_loss, optimizer = optimizer)

batch_size = 8192
learning_rates = [1e-3, 1e-3, 1e-4, 1e-4, 1e-5, 1e-5, 1e-5, 1e-5]
T_c = 5

for l in range(len(learning_rates)):
	print("\nTraining Session ", l + 1, "\nT_c =", T_c)
	triplets = generate_triplets(150000, T_c)
	optimizer.learning_rate.assign(learning_rates[l])
	print("\nBatch Size: ", batch_size, "\nEpochs: ", 10, "\nLearning rate: ", learning_rates[l])
	triplets_concat = tf.concat(triplets, axis=1)
	model.fit(triplets, triplets_concat, batch_size = batch_size, epochs = 10)


Training Session  1 
T_c = 5
Generating lookup table for positive samples
Loading batch of triplets from dataset (on disk or in RAM)
Finished loading triplet batch

Batch Size:  8192 
Epochs:  10 
Learning rate:  0.001
Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 21s 876ms/step - loss: 0.8745
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 17s 767ms/step - loss: 0.1587
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 15s 688ms/step - loss: 0.0982
Epoch 4/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 13s 663ms/step - loss: 0.0705
Epoch 5/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 13s 663ms/step - loss: 0.0545
Epoch 6/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 13s 662ms/step - loss: 0.0461
Epoch 7/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 13s 664ms/step - loss: 0.0416
Epoch 8/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 13s 663ms/step - loss: 0.0419
Epoch 9/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 13s 663ms/step - loss: 0.0408
Epoch 10/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 13s 667ms/step - loss: 0.0422

Training Session  2 
T_c = 5
Generating lookup table for positive samples
Loading batch of triplet

# Plotting the Channel Chart

In [32]:
from pathlib import Path

# Predict the triplet baseline chart positions with the trained embedding model.
channel_chart_positions = []
for csi, pos, timestamp in training_set.batch(1000):
	channel_chart_positions.append(embedding_model.predict(csi, verbose=0))

channel_chart_positions = np.vstack(channel_chart_positions).astype(np.float32)
groundtruth_positions = groundtruth_positions.astype(np.float32)
print(len(groundtruth_positions))
print(groundtruth_positions.shape)

20997
(20997, 2)


In [33]:
plot_colorized(channel_chart_positions, title = "Channel Chart")


# Unified Evaluation, Plotting, and Saving

In [34]:
import random
from pathlib import Path

from common_metrics import chart_metrics, localization_error_summary, write_json
from run_adp_siamese import plot_chart


In [35]:
notebook_stride = 4 # 点来自原始 tfrecords 每 4 个取 1 
metric_seed = 10
metric_subset_fraction = 0.2
run_name = f"stride{notebook_stride}_n{len(groundtruth_positions)}_seed{SEED}"
out_dir = Path("results/triplet_baseline_notebook") / run_name
out_dir.mkdir(parents=True, exist_ok=True)

subset_size = max(1, int(len(groundtruth_positions) * metric_subset_fraction))
subset_indices = random.Random(metric_seed).sample(range(len(groundtruth_positions)), subset_size)
metrics = chart_metrics(groundtruth_positions[subset_indices], channel_chart_positions[subset_indices])
errors = localization_error_summary(groundtruth_positions, channel_chart_positions)

indices = np.arange(0, len(groundtruth_positions) * notebook_stride, notebook_stride, dtype=np.int64)
result = {
    "method": "Triplet baseline from ChannelCharting.ipynb",
    "dataset": "datasets/dichasus-cf02-cf03-cf04.tfrecords",
    "run_name": run_name,
    "num_samples": int(len(groundtruth_positions)),
    "stride": int(notebook_stride),
    "seed": int(SEED),
    "metric_seed": int(metric_seed),
    "metric_subset_fraction": float(metric_subset_fraction),
    "metric_subset_size": int(subset_size),
    "baseline_source": "ChannelCharting.ipynb",
    "input_format": "DICHASUS tfrecords",
    "feature_note": "Notebook tfrecords pipeline: calibration, subcarrier average 508:516, R2M real feature.",
    "positive_sampling": "timestamp window Tc",
    "negative_sampling": "random sample from the notebook triplet generator",
    "metrics": metrics,
    "localization_errors_after_affine": errors,
    "elapsed_s": None,
}

np.savez(out_dir / "embedding.npz", indices=indices, positions=groundtruth_positions, embedding=channel_chart_positions)
write_json(out_dir / "result.json", result)

# Keep the original tutorial visualization for sanity checking the triplet chart shape.
plot_colorized(channel_chart_positions, title="Channel Chart")
plt.savefig(out_dir / "chart_raw.png", dpi=180, bbox_inches="tight")
plt.close()

# Save the unified comparison plot used by the ADP/Siamese scripts.
# This plot affinely aligns the embedding to ground-truth coordinates before drawing.
plot_chart(
    out_dir / "chart.png",
    groundtruth_positions,
    channel_chart_positions,
    f"Triplet Baseline\nCT={metrics['ct']:.3f}, TW={metrics['tw']:.3f}, KS={metrics['ks']:.3f}",
)
embedding_model.save(out_dir / "embedding_model.keras")

print("RESULT", json.dumps(result, indent=2, sort_keys=True))


RESULT {
  "baseline_source": "ChannelCharting.ipynb",
  "dataset": "datasets/dichasus-cf02-cf03-cf04.tfrecords",
  "elapsed_s": null,
  "feature_note": "Notebook tfrecords pipeline: calibration, subcarrier average 508:516, R2M real feature.",
  "input_format": "DICHASUS tfrecords",
  "localization_errors_after_affine": {
    "max_m": 6.57588529586792,
    "mean_m": 1.6476918458938599,
    "median_m": 1.3055118322372437,
    "p90_m": 3.527855396270752,
    "p95_m": 4.138086795806885
  },
  "method": "Triplet baseline from ChannelCharting.ipynb",
  "metric_seed": 10,
  "metric_subset_fraction": 0.2,
  "metric_subset_size": 4199,
  "metrics": {
    "ct": 0.9754647776156656,
    "ks": 0.2647391245535612,
    "n_neighbors": 209,
    "tw": 0.9595701537040954
  },
  "negative_sampling": "random sample from the notebook triplet generator",
  "num_samples": 20997,
  "positive_sampling": "timestamp window Tc",
  "run_name": "stride4_n20997_seed10",
  "seed": 10,
  "stride": 4
}
